### Cell 0
# 実験用の参考文献文字列の作成
* 予備実験用
* 評価実験用

### Cell 1
## 日本学術会議協力学術研究団体
* 登録団体の論文からメタデータを抽出し、参考文献文字列を生成する
* 登録団体一覧→scj_registered_AcademicSocieties.txt

In [1]:
# Cell 2
with open("../5データ/5-3日本学術会議協力学術研究団体/scj_registered_AcademicSocieties.txt","r") as f:
    lines = f.readlines()

num_of_AcademicSocieties = 0
for line in lines:
    academic_societies = line.split(",")
    num_of_AcademicSocieties += len(academic_societies)

print(f"日本学術会議登録団体数：{num_of_AcademicSocieties}")

日本学術会議登録団体数：2195


### Cell 3
## <font color="red">要実行</font>

In [1]:
# Cell 4
# モジュール・接続文字列の読み込み
import pymongo
import re
import random
import copy
url_mongodb = "mongodb://localhost:27017/"

# mongodbへの接続
myclient = pymongo.MongoClient(url_mongodb)
mydb = myclient["jalc"]
mycol = mydb["restapi"]

### Cell 5
## 事前準備(実行しなくていい)

In [3]:
# Cell 6
#日本語を含むか判定
def has_japanese(text):
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

# クエリ
query = {"$and":[
    {"publisher_list":{"$exists":True}},
    {"publisher_list.publisher_name":{"$regex":" "}}
]}

publisher_list = []
for doc in mycol.find(query,{"_id":0}):
    for publisher in doc["publisher_list"]:
        if has_japanese(publisher["publisher_name"]):
            if publisher["publisher_name"] not in publisher_list:
                publisher_list.append(publisher["publisher_name"])

for i in publisher_list:
    print(i)                


公益社団法人 日本気象学会
日本デジタルゲーム学会
公益社団法人 日本補綴歯科学会
公益社団法人 日本獣医師会
社団法人 日本作業療法士協会
岡山医学会
一般社団法人 日本教育学会
一般社団法人 日本経カテーテル心臓弁治療学会
一般社団法人 日本集中治療医学会
日本教育社会学会
特定非営利活動法人 日本心臓血管外科学会
日本食品照射研究協議会
公益社団法人 日本実験動物学会
獣医疫学会
公益社団法人 日本繁殖生物学会
日本繁殖生物学会
公益社団法人 日本地すべり学会
一般社団法人 日本膵臓学会
日本茶業学会
FORMATH研究学会
情報知識学会
公益社団法人 日本水環境学会
国際光治療学会
公益財団法人 山階鳥類研究所
日本組織適合性学会
公益社団法人 日本栄養・食糧学会
社団法人 日本産業衛生学会
公益社団法人 日本産業衛生学会
一般社団法人 日本地質学会
一般社団法人 日本感染症学会
日本高圧力学会
国際観光学研究会
公益社団法人 日本植物学会
日本臨床外科学会
日本感情心理学会
一般社団法人 日本輸血・細胞治療学会
日本不安症学会
日本不安障害学会
日本宇宙生物科学会
日本応用動物昆虫学会
一般社団法人 照明学会
日本良導絡自律神経学会
日本官能評価学会
国立大学法人　徳島大学医学部
日本ヘーゲル学会
公益社団法人 日本航海学会
一般社団法人 日本医療機器学会
一般社団法人日本医療機器学会
一般社団法人 日本農村医学会
日本緬羊研究会
公益財団法人 天理よろづ相談所　医学研究所
一般社団法人 日本風工学会
特定非営利活動法人国際応用情報学研究機構
一般社団法人 日本呼吸器内視鏡学会
日本食生活学会
一般社団法人 日本環境化学会
一般社団法人 プラスチック成形加工学会
一般社団法人 日本内分泌学会
Annals of Thoracic and Cardiovascular Surgery 編集委員会
比較経済体制学会
日本レーザー歯学会
公益社団法人 日本薬学会
公益社団法人 日本木材保存協会
日本クリティカルケア看護学会
特定非営利活動法人 日本ブリーフセラピー協会
公益財団法人 鉄道総合技術研究所
岡山大学大学院環境生命科学研究科
公益財団法人 医療科学研究所
パワーエレクトロニクス学会
一般社団法人日本教育工学会・一般社団法人教育システム情報学会
教育シス

In [30]:
# Cell 7
# クエリ
query = {
    "$and":[
        {"content_type":"JA"},
        {"publisher_list":{"$exists":True}},
        {"publication_date.publication_year":"2020"},
        {"title_list":{"$exists":True}},
        {"creator_list":{"$exists":True}},
        {"publication_date":{"$exists":True}},
        {"journal_title_name_list":{"$exists":True}},
        {"volume":{"$exists":True}},
        {"issue":{"$exists":True}},
        {"first_page":{"$exists":True}},
        {"last_page":{"$exists":True}},
        {"$or":[
            {"issue":"1"},
            {"issue":"2"}
        ]}
    ]
}

# 日本学術会議登録団体のリスト
scj_registered_AcademicSocieties = []
with open("../5データ/5-3日本学術会議協力学術研究団体/scj_registered_AcademicSocieties.txt","r") as f:
    lines = f.readlines()
for line in lines:
    tmp = line.split(",")
    scj_registered_AcademicSocieties += tmp

result_num = 0  # 該当件数    

# MongoDBでの検索
for doc in mycol.find(query):
    publisher_list = doc.get("publisher_list","")
    flag = False
    for val in publisher_list:
        publisher_name = val["publisher_name"]
        for academic_societie in scj_registered_AcademicSocieties:
            if academic_societie in publisher_name:
                flag = True
                break
        if flag:
            break
    if flag:
        result_num += 1        

print(result_num)

16461


### Cell 8
## ”参考文献文字列（実在）_予備実験用”の作成
* フォーマットとして人工知能学会、情報処理学会、日本言語学会を採用
* 誤植を含むものも参考文献文字列（実在）として扱う

In [ ]:
# Cell 9
# MongoDB クエリ
query = {
    "$and":[
        {"content_type":"JA"},
        {"title_list":{"$exists":True}},
        {"creator_list":{"$exists":True}},
        {"publication_date":{"$exists":True}},
        {"journal_title_name_list":{"$exists":True}},
        {"volume":{"$exists":True}},
        {"issue":{"$exists":True}},
        {"first_page":{"$exists":True}},
        {"last_page":{"$exists":True}},
        {"publisher_list":{"$exists":True}},
        {"publication_date.publication_year":"2020"},
        {"$or":[
            {"volume":"1"},
            {"volume":"2"},
            {"volume":"3"},
            {"volume":"4"}
        ]}  
    ]
}

# 日本学術会議登録団体のリスト
scj_registered_AcademicSocieties = []
with open("../5データ/5-3日本学術会議協力学術研究団体/scj_registered_AcademicSocieties.txt","r") as f:
    lines = f.readlines()
for line in lines:
    tmp = line.split(",")
    scj_registered_AcademicSocieties += tmp           

In [4]:
# Cell 10
# 日本語を含むか判定
def has_japanese(text):
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

# 文字列内のランダムな一字を除去する関数
def remove_random_char(s):
    if not s:
        return ""  # 文字列が空の場合は空の文字列を返す

    # 文字列の長さからランダムなインデックスを生成
    random_index = random.randint(0, len(s) - 1)
    
    # スライスを使ってランダムな文字を除去
    # random_indexの前の部分と、後の部分を結合する
    return s[:random_index] + s[random_index + 1:]

# 標題・著者・雑誌名に日本語があるか判定
def check_lang(doc):
    title_lang,creator_lang,journal_lang = "","",""
    # 表題の言語について
    for title in doc["title_list"]:
        lang = title.get("lang","")
        if lang == "":  # lang属性を持たない場合
            if has_japanese(title.get("title","")):
                lang = "ja"
        if lang == "ja":
            title_lang = "ja"
            break
    # 著者の言語について
    for creator in doc["creator_list"]:
        for creator_name in creator["names"]:
            lang = creator_name.get("lang","")
            if lang == "":  # lang属性を持たない場合
                if has_japanese(creator_name.get("first_name","")):
                    lang = "ja"
            if lang == "ja":
                creator_lang = "ja"
                break
    # 雑誌の言語について
    for journal in doc["journal_title_name_list"]:
        lang = journal.get("lang","")
        if lang == "":
            if has_japanese(journal.get("journal_title_name","")):
                lang = "ja"
        if lang == "ja":
            journal_lang = "ja"
            break
    return title_lang == "ja" and creator_lang == "ja" and journal_lang == "ja"                                        

# 学術会議登録団体であるか判定(計算量が多いの改善)
def check_publisher(doc,scj_registered_AcademicSocieties):
    flag = False
    for academicSocieties in scj_registered_AcademicSocieties:
        for publisher in doc["publisher_list"]:
            if academicSocieties in publisher.get("publisher_name",""):
                flag = True
    return flag            

# 標題の取得
def get_title(doc):
    title_text = ""
    for val in doc["title_list"]:
        lang,title,subtitle = val.get("lang",""),val.get("title",""),val.get("subtitle","")
        if lang == "":
            if has_japanese(title):
                lang = "ja"
        if lang == "ja":
            title_text = title
            if subtitle != "":
                title_text = title_text + " " + subtitle
    return title_text                   

# 雑誌名の取得
def get_journal(doc):
    journal_text = ""
    for val in doc["journal_title_name_list"]:
        journal_title,Type,lang = val.get("journal_title_name",""),val.get("type",""),val.get("lang","")
        if lang == "":
            if has_japanese(journal_title):
                lang = "ja"
        if lang == "ja":
            journal_text = journal_title
            if Type == "full":
                break
    return journal_text                

# 著者の取得
def get_creator(doc):
    creator_list = []
    for val in doc["creator_list"]:
        if len(val["names"]) == 1:  # 著者の表記が1通りの場合
            last_name,first_name = val["names"][0].get("last_name",""),val["names"][0].get("first_name","")
            creator_list.append([last_name,first_name])
        else:    # 著者の表記が2通り以上の場合
            for x in val["names"]:
                last_name,first_name,lang = x.get("last_name",""),x.get("first_name",""),x.get("lang","")
                if has_japanese(first_name):
                    lang = "ja"
                if lang == "ja":
                    creator_list.append([last_name,first_name])
                    break
    return creator_list            

# 参考文献文字列の作成(人工知能学会)
def create_reference_jsai(creator_list,title,journal,volume,issue,page_list,year):
    tmp = []
    error_probability = random.randint(0,99)   # 誤植を含むか否かの乱数(含む確率35%)
    for name in creator_list:
            if name[0] != "":
                if has_japanese(name[0]):
                    tmp.append(name[0])
                else:
                    tmp.append(name[0]+", "+name[1][0]+".")
            else:
                tmp.append(name[1])
    if error_probability >= 35:    # 誤植を含まない場合            
        creator_text = "，".join(tmp)
        reference = creator_text + "：" + title + "，" + journal + "，" + "Vol." + volume + ", " + "No."\
        + issue + ", " + "pp." + page_list[0] + "-" + page_list[1] + " " + "(" + year + ")" + "."
    else:   # 誤植を含まない場合
        error_num_probability = random.randint(0,99)    # 誤植をいくつ含むか(1:80%, 2:20%)
        bib_elements_probability = random.randint(0,99) # どの書誌要素に誤植を含ませるか
        # 誤植が1つ
        if bib_elements_probability < 40:   # タイトルに誤植を挿入
            title = remove_random_char(title)
        elif bib_elements_probability < 60: # 著者一名の脱落
            if len(tmp) != 1:
                tmp.pop(-1)
        elif bib_elements_probability < 75: # 雑誌名のランダムな一字の脱落
            journal = remove_random_char(journal)
        elif bib_elements_probability < 80 and year.isdigit(): # 出版年の値を±1
            df = random.choice([1,-1])
            year = int(year) + df
            year = str(year)
        elif bib_elements_probability < 85 and volume.isdigit() : # 巻の値を±1
            df = random.choice([1,-1])
            volume = int(volume) + df
            volume = str(volume)
        elif bib_elements_probability < 90 and issue.isdigit(): # 号の値を±1
            df = random.choice([1,-1])
            issue = int(issue) + df
            issue = str(issue)
        else:
            df = random.choice([1,-1])
            random_index = random.choice([0,1])
            if page_list[random_index].isdigit():
                page_list[random_index] = int(page_list[random_index]) + df
                page_list[random_index] = str(page_list[random_index])

        if error_num_probability < 20:  # 誤植が２つの場合
            bib_elements_probability = random.randint(0,99)
            if bib_elements_probability < 40:   # タイトルに誤植を挿入
                title = remove_random_char(title)
            elif bib_elements_probability < 60: # 著者一名の脱落
                if len(tmp) != 1:
                    tmp.pop(-1)
            elif bib_elements_probability < 75: # 雑誌名のランダムな一字の脱落
                journal = remove_random_char(journal)
            elif bib_elements_probability < 80 and year.isdigit(): # 出版年の値を±1
                df = random.choice([1,-1])
                year = int(year) + df
                year = str(year)
            elif bib_elements_probability < 85 and volume.isdigit() : # 巻の値を±1
                df = random.choice([1,-1])
                volume = int(volume) + df
                volume = str(volume)
            elif bib_elements_probability < 90 and issue.isdigit(): # 号の値を±1
                df = random.choice([1,-1])
                issue = int(issue) + df
                issue = str(issue)
            else:
                df = random.choice([1,-1])
                random_index = random.choice([0,1])
                if page_list[random_index].isdigit():
                    page_list[random_index] = int(page_list[random_index]) + df
                    page_list[random_index] = str(page_list[random_index])
        
        creator_text = "，".join(tmp)
        reference = creator_text + "：" + title + "，" + journal + "，" + "Vol." + volume + ", " + "No."\
        + issue + ", " + "pp." + page_list[0] + "-" + page_list[1] + " " + "(" + year + ")" + "."                       
    return reference

# 参考文献文字列の作成(情報処理学会)
def create_reference_ipsj(creator_list,title,journal,volume,issue,page_list,year,count):
    creator_text = ""
    error_probability = random.randint(0,99)    # 誤植を含むか否かの乱数(含む確率35%)
    tmp = []
    num = 0
    for name in creator_list:
        num += 1
        if num == 4:
            creator_text = "ほか"
            break
        else:
            if name[0] != "":
                if has_japanese(name[0]):
                    tmp.append(name[0] + name[1])
                else:
                    tmp.append(name[0]+", "+name[1][0]+".")
            else:
                tmp.append(name[1])
    if error_probability >= 35: # 誤植を含まない場合           
        creator_text = "，".join(tmp) + creator_text
        reference = creator_text+"："+title+"，"+journal+"，"+"Vol."+volume+"，"+"No."+issue+"，"+"pp."+page_list[0]+"-"+\
        page_list[1]+"（"+year+"）"+"．"
    else:   # 誤植を含む場合
        error_num_probability = random.randint(0,99)    # 誤植をいくつ含むか(1:80%, 2:20%)
        bib_elements_probability = random.randint(0,99) # どの書誌要素に誤植を含ませるか
        # 誤植が1つ
        if bib_elements_probability < 40:   # タイトルに誤植を挿入
            title = remove_random_char(title)
        elif bib_elements_probability < 60: # 著者一名の脱落
            if len(tmp) != 1:
                tmp.pop(-1)
        elif bib_elements_probability < 75: # 雑誌名のランダムな一字の脱落
            journal = remove_random_char(journal)
        elif bib_elements_probability < 80 and year.isdigit(): # 出版年の値を±1
            df = random.choice([1,-1])
            year = int(year) + df
            year = str(year)
        elif bib_elements_probability < 85 and volume.isdigit() : # 巻の値を±1
            df = random.choice([1,-1])
            volume = int(volume) + df
            volume = str(volume)
        elif bib_elements_probability < 90 and issue.isdigit(): # 号の値を±1
            df = random.choice([1,-1])
            issue = int(issue) + df
            issue = str(issue)
        else:
            df = random.choice([1,-1])
            random_index = random.choice([0,1])
            if page_list[random_index].isdigit():
                page_list[random_index] = int(page_list[random_index]) + df
                page_list[random_index] = str(page_list[random_index])

        if error_num_probability < 20:  # 誤植が２つの場合
            bib_elements_probability = random.randint(0,99)
            if bib_elements_probability < 40:   # タイトルに誤植を挿入
                title = remove_random_char(title)
            elif bib_elements_probability < 60: # 著者一名の脱落
                if len(tmp) != 1:
                    tmp.pop(-1)
            elif bib_elements_probability < 75: # 雑誌名のランダムな一字の脱落
                journal = remove_random_char(journal)
            elif bib_elements_probability < 80 and year.isdigit(): # 出版年の値を±1
                df = random.choice([1,-1])
                year = int(year) + df
                year = str(year)
            elif bib_elements_probability < 85 and volume.isdigit() : # 巻の値を±1
                df = random.choice([1,-1])
                volume = int(volume) + df
                volume = str(volume)
            elif bib_elements_probability < 90 and issue.isdigit(): # 号の値を±1
                df = random.choice([1,-1])
                issue = int(issue) + df
                issue = str(issue)
            else:
                df = random.choice([1,-1])
                random_index = random.choice([0,1])
                if page_list[random_index].isdigit():
                    page_list[random_index] = int(page_list[random_index]) + df
                    page_list[random_index] = str(page_list[random_index])
        if len(tmp) <= 3:
            creator_text = ""
        creator_text = "，".join(tmp) + creator_text
        reference = creator_text+"："+title+"，"+journal+"，"+"Vol."+volume+"，"+"No."+issue+"，"+"pp."+page_list[0]+"-"+\
        page_list[1]+"（"+year+"）"+"．"   
    return reference         

# 参考文献文字列の作成(日本言語学会)
def create_reference_lsj(creator_list,title,journal,volume,issue,page_list,year):
    error_probability = random.randint(0,99)
    tmp = []
    for name in creator_list:
        tmp.append(name[0]+name[1])
    if error_probability >= 35: # 誤植を含まない場合    
        reference = "・".join(tmp)+"（"+year+"）"+"「"+title+"」"+"『"+journal+"』"+volume+"（"+issue+"）"+": "+page_list[0]+"–"+page_list[1]+"."
    else:   # 誤植を含む場合
        error_num_probability = random.randint(0,99)    # 誤植をいくつ含むか(1:80%, 2:20%)
        bib_elements_probability = random.randint(0,99) # どの書誌要素に誤植を含ませるか
        # 誤植が1つ
        if bib_elements_probability < 40:   # タイトルに誤植を挿入
            title = remove_random_char(title)
        elif bib_elements_probability < 60: # 著者一名の脱落
            if len(tmp) != 1:
                tmp.pop(-1)
        elif bib_elements_probability < 75: # 雑誌名のランダムな一字の脱落
            journal = remove_random_char(journal)
        elif bib_elements_probability < 80 and year.isdigit(): # 出版年の値を±1
            df = random.choice([1,-1])
            year = int(year) + df
            year = str(year)
        elif bib_elements_probability < 85 and volume.isdigit() : # 巻の値を±1
            df = random.choice([1,-1])
            volume = int(volume) + df
            volume = str(volume)
        elif bib_elements_probability < 90 and issue.isdigit(): # 号の値を±1
            df = random.choice([1,-1])
            issue = int(issue) + df
            issue = str(issue)
        else:
            df = random.choice([1,-1])
            random_index = random.choice([0,1])
            if page_list[random_index].isdigit():
                page_list[random_index] = int(page_list[random_index]) + df
                page_list[random_index] = str(page_list[random_index])

        if error_num_probability < 20:  # 誤植が２つの場合
            bib_elements_probability = random.randint(0,99)
            if bib_elements_probability < 40:   # タイトルに誤植を挿入
                title = remove_random_char(title)
            elif bib_elements_probability < 60: # 著者一名の脱落
                if len(tmp) != 1:
                    tmp.pop(-1)
            elif bib_elements_probability < 75: # 雑誌名のランダムな一字の脱落
                journal = remove_random_char(journal)
            elif bib_elements_probability < 80 and year.isdigit(): # 出版年の値を±1
                df = random.choice([1,-1])
                year = int(year) + df
                year = str(year)
            elif bib_elements_probability < 85 and volume.isdigit() : # 巻の値を±1
                df = random.choice([1,-1])
                volume = int(volume) + df
                volume = str(volume)
            elif bib_elements_probability < 90 and issue.isdigit(): # 号の値を±1
                df = random.choice([1,-1])
                issue = int(issue) + df
                issue = str(issue)
            else:
                df = random.choice([1,-1])
                random_index = random.choice([0,1])
                if page_list[random_index].isdigit():
                    page_list[random_index] = int(page_list[random_index]) + df
                    page_list[random_index] = str(page_list[random_index])
        
        reference = "・".join(tmp)+"（"+year+"）"+"「"+title+"」"+"『"+journal+"』"+volume+"（"+issue+"）"+": "+page_list[0]+"–"+page_list[1]+"."

    return reference     

# 以下本処理

documents = mycol.find(query)
title_list = []
count = 1
reference_jsai_list = []
reference_ipsj_list = []
reference_lsj_list = []
doi_list = []

for doc in documents:
    if check_lang(doc) and check_publisher(doc,scj_registered_AcademicSocieties):
        title = get_title(doc)
        #title_list.append(title)
        journal = get_journal(doc)
        volume = doc["volume"]
        issue = doc["issue"]
        page_list = [doc["first_page"],doc["last_page"]]
        year = doc["publication_date"]["publication_year"]
        creator_list = get_creator(doc)
        # 人工知能学会スタイルでの作成
        tmp_title = title
        tmp_journal = journal
        tmp_year = year
        tmp_volume = volume
        tmp_issue = issue
        reference_jsai = create_reference_jsai(creator_list.copy(),tmp_title,tmp_journal,tmp_volume,tmp_issue,page_list.copy(),tmp_year)
        # 情報処理学会スタイルでの作成
        tmp_title = title
        tmp_journal = journal
        tmp_year = year
        tmp_volume = volume
        tmp_issue = issue
        reference_ipsj = create_reference_ipsj(creator_list.copy(),tmp_title,tmp_journal,tmp_volume,tmp_issue,page_list.copy(),tmp_year,count)
        # 日本言語学会スタイルでの作成
        tmp_title = title
        tmp_journal = journal
        tmp_year = year
        tmp_volume = volume
        tmp_issue = issue
        reference_lsj = create_reference_lsj(creator_list.copy(),tmp_title,tmp_journal,tmp_volume,tmp_issue,page_list.copy(),tmp_year)
        count += 1
        reference_jsai_list.append(reference_jsai)
        reference_ipsj_list.append(reference_ipsj)
        reference_lsj_list.append(reference_lsj)
        """
        doi = doc["doi"]
        doi_list.append(doi)
        print(reference_jsai)
        print(reference_ipsj)
        print(reference_lsj)
        print("="*6)
        if count == 101:
            break
        """    

# テキストファイルの作成

with open('../5データ/5-1予備実験用データ/5-1-1参考文献文字列（実在）/positive_reference_jsai_pre.txt','x') as f:
    f.write('\n'.join(reference_jsai_list))
with open('../5データ/5-1予備実験用データ/5-1-1参考文献文字列（実在）/positive_reference_ipsj_pre.txt','x') as f:
    f.write('\n'.join(reference_ipsj_list))
with open('../5データ/5-1予備実験用データ/5-1-1参考文献文字列（実在）/positive_reference_lsj_pre.txt','x') as f:
    f.write('\n'.join(reference_lsj_list))
"""
print(len(doi_list))
with open('../5データ/5-1予備実験用データ/5-1-1参考文献文字列（実在）/doi_list.txt','x') as f:
    f.write('\n'.join(doi_list))
"""                                                  

"\nprint(len(doi_list))\nwith open('pre_experiment_data/positive_example/doi_list.txt','x') as f:\n    f.write('\n'.join(doi_list))\n"

### Cell 11
## ”参考文献文字列（架空）_予備実験用”の作成
* フォーマットは参考文献文字列（実在）と同様
* 誤植は発生させていない

In [11]:
# Cell 12
# 標題をテキストファイルに書き込む
with open('../5データ/5-1予備実験用データ/5-1-3論文タイトル一覧/pre_experiment_titles.txt','x') as f:
    f.write('\n'.join(title_list))

In [4]:
# Cell 13
# Geminiが生成した表題をリストにまとめる
with open('../5データ/5-1予備実験用データ/5-1-3論文タイトル一覧/llm_create_titles_2.txt','r') as f:
    lines = f.readlines()

# テキストファイルには改行コードが含まれているためそれを除去
cleaned_lines = []
for line in lines:
    cleaned_lines.append(line.strip())
llm_create_titles = []
for cleaned_line in cleaned_lines:
    llm_create_titles.append(cleaned_line)

for i in range(10):
    print(llm_create_titles[i])    

print(f"表題：{len(llm_create_titles)}件")

ゴルフ場の活性化に向けたサービス・ドミナント・ロジックの応用 ― 価値共創概念の新たな価値の創出 ―
グローバル経営における国民文化の影響力分析 ― 経営哲学の国別比較を通じて ―
顧客志向と販売志向が営業担当者の客観的業績に与える影響 ― アパレル業界での実証調査 ―
リードユーザーとしての消費者の特徴に関する調査に基づく実証研究 ― リードユーザー性の先行要因と結果 ―
ユーザーが創造した製品の情報提示と制御焦点理論 ― オンライン実験による仲介分析 ―
移動中の人々の洞察 ― 移動する生活者の行動と心理に関する研究 ―
居住地選好アンケートデータを用いた住みたい地域の抽出
患者・主介護者との強固な信頼関係構築のため医師に求められる対話の考察 ― 医師、患者・主介護者の適合性 ―
サービスチェーンにおける非正規従業員の働く意欲が業績につながる仕組み ― 飲食チェーン3社での実証研究 ―
対立する論理の協調による消費行動の脱スティグマ化 ― 「婚活」ブームを事例として ―
表題：712件


In [ ]:
# Cell 14
# 参考文献文字列　予備実験　正例の作成クエリを実行している必要あり

# 日本語を含むか判定
def has_japanese(text):
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

# 標題・著者・雑誌名に日本語があるか判定
def check_lang(doc):
    title_lang,creator_lang,journal_lang = "","",""
    # 表題の言語について
    for title in doc["title_list"]:
        lang = title.get("lang","")
        if lang == "":  # lang属性を持たない場合
            if has_japanese(title.get("title","")):
                lang = "ja"
        if lang == "ja":
            title_lang = "ja"
            break
    # 著者の言語について
    for creator in doc["creator_list"]:
        for creator_name in creator["names"]:
            lang = creator_name.get("lang","")
            if lang == "":  # lang属性を持たない場合
                if has_japanese(creator_name.get("first_name","")):
                    lang = "ja"
            if lang == "ja":
                creator_lang = "ja"
                break
    # 雑誌の言語について
    for journal in doc["journal_title_name_list"]:
        lang = journal.get("lang","")
        if lang == "":
            if has_japanese(journal.get("journal_title_name","")):
                lang = "ja"
        if lang == "ja":
            journal_lang = "ja"
            break
    return title_lang == "ja" and creator_lang == "ja" and journal_lang == "ja"                                        

# 学術会議登録団体であるか判定(計算量が多いの改善)
def check_publisher(doc,scj_registered_AcademicSocieties):
    flag = False
    for academicSocieties in scj_registered_AcademicSocieties:
        for publisher in doc["publisher_list"]:
            if academicSocieties in publisher.get("publisher_name",""):
                flag = True
    return flag            

# 標題の取得
def get_title(doc):
    title_text = ""
    for val in doc["title_list"]:
        lang,title,subtitle = val.get("lang",""),val.get("title",""),val.get("subtitle","")
        if lang == "":
            if has_japanese(title):
                lang = "ja"
        if lang == "ja":
            title_text = title
            if subtitle != "":
                title_text = title_text + " " + subtitle
    return title_text                   

# 雑誌名の取得
def get_journal(doc):
    journal_text = ""
    for val in doc["journal_title_name_list"]:
        journal_title,Type,lang = val.get("journal_title_name",""),val.get("type",""),val.get("lang","")
        if lang == "":
            if has_japanese(journal_title):
                lang = "ja"
        if lang == "ja":
            journal_text = journal_title
            if Type == "full":
                break
    return journal_text                

# 著者の取得
def get_creator(doc):
    creator_list = []
    for val in doc["creator_list"]:
        if len(val["names"]) == 1:  # 著者の表記が1通りの場合
            last_name,first_name = val["names"][0].get("last_name",""),val["names"][0].get("first_name","")
            creator_list.append([last_name,first_name])
        else:    # 著者の表記が2通り以上の場合
            for x in val["names"]:
                last_name,first_name,lang = x.get("last_name",""),x.get("first_name",""),x.get("lang","")
                if has_japanese(first_name):
                    lang = "ja"
                if lang == "ja":
                    creator_list.append([last_name,first_name])
                    break
    return creator_list            

# 学会名の取得
def get_publisher(doc,scj_registered_AcademicSocieties):
    publisher_name = ""
    for academicSocieties in scj_registered_AcademicSocieties:
        finish_flag = False
        for publisher in doc["publisher_list"]:
            if academicSocieties in publisher.get("publisher_name",""):
                publisher_name = academicSocieties
                finish_flag = True
        if finish_flag:
            break
    return publisher_name            

# 参考文献文字列の作成(人工知能学会)
def create_reference_jsai(creator_list,llm_create_title,publisher_name,year):
    tmp = []
    for name in creator_list:
        if name[0] != "":
            if has_japanese(name[0]):
                tmp.append(name[0])
            else:
                tmp.append(name[0]+", "+name[1][0]+".")
        else:
            tmp.append(name[1])                
    if len(tmp) != 1:   # 著者を一名捨てる
        tmp.pop(-1)            
    creator_text = "，".join(tmp)
    year = int(year) - 1
    reference = creator_text + "：" + llm_create_title + "，" + publisher_name + " " + "(" + str(year) + ")" + "."
    return reference

# 参考文献文字列の作成(情報処理学会)
def create_reference_ipsj(creator_list,llm_create_title,publisher_name,year):
    creator_text = ""
    tmp = []
    num = 0
    for name in creator_list:
        num += 1
        if num == 5:
            creator_text = "ほか"
            break
        else:
            if name[0] != "": 
                if has_japanese(name[0]):
                    tmp.append(name[0] + name[1])
                else:
                    tmp.append(name[0]+", "+name[1][0]+".")
            else:
                tmp.append(name[1])
    if len(tmp) != 1:   # 著者一名を削除
        tmp.pop(-1)                    
    creator_text = "，".join(tmp) + creator_text
    year = int(year) - 1
    reference = creator_text+"："+llm_create_title+"，"+publisher_name+" "+"（"+str(year)+"）"+"．"
    return reference         

# 参考文献文字列の作成(日本言語学会)
def create_reference_lsj(creator_list,llm_create_title,publisher,year):
    tmp = []
    for name in creator_list:
        tmp.append(name[0]+name[1])
    if len(tmp) != 1:
        tmp.pop(-1)
    year = int(year) - 1        
    reference = "・".join(tmp)+"（"+str(year)+"）"+"「"+llm_create_title+"」"+"『"+publisher+"』"
    return reference     

# 以下本処理

documents = mycol.find(query)
count = 0
negative_examples_num = 0
reference_jsai_list = []
reference_ipsj_list = []
reference_lsj_list = []

for doc in documents:
    if check_lang(doc) and check_publisher(doc,scj_registered_AcademicSocieties):
        llm_create_title = llm_create_titles[count]
        title = get_title(doc)
        journal = get_journal(doc)
        volume = doc["volume"]
        issue = doc["issue"]
        page_list = [doc["first_page"],doc["last_page"]]
        year = doc["publication_date"]["publication_year"]
        creator_list = get_creator(doc)
        publisher_name = get_publisher(doc,scj_registered_AcademicSocieties)
        
        #reference_jsai = create_reference_jsai(creator_list,llm_create_title,publisher_name,year)
        #reference_ipsj = create_reference_ipsj(creator_list,llm_create_title,publisher_name,year)
        #reference_lsj = create_reference_lsj(creator_list,title,journal,volume,issue,page_list,year)

        if title != llm_create_title:   # Geminiによる変換が上手くいっているもの [if llm_create_title not in title:]に変換したほうがいいかも
            negative_examples_num += 1
            reference_jsai = create_reference_jsai(creator_list,llm_create_title,publisher_name,year)
            reference_ipsj = create_reference_ipsj(creator_list,llm_create_title,publisher_name,year)
            reference_lsj = create_reference_lsj(creator_list,llm_create_title,publisher_name,year)
            reference_jsai_list.append(reference_jsai)
            reference_ipsj_list.append(reference_ipsj)
            reference_lsj_list.append(reference_lsj)
        count += 1

# テキストファイルの作成        
with open('../5データ/5-1予備実験用データ/5-1-2参考文献文字列（架空）/negative_reference_jsai_pre.txt','x') as f:
    f.write('\n'.join(reference_jsai_list))
with open('../5データ/5-1予備実験用データ/5-1-2参考文献文字列（架空）/negative_reference_ipsj_pre.txt','x') as f:
    f.write('\n'.join(reference_ipsj_list))
with open('../5データ/5-1予備実験用データ/5-1-2参考文献文字列（架空）/negative_reference_lsj_pre.txt','x') as f:
    f.write('\n'.join(reference_lsj_list))        
                                              

### Cell 15
## ”参考文献文字列（実在）_評価実験用”の作成
* 予備実験とは異なるデータ

In [2]:
# Cell 16
# 評価実験用 DOI の読み込み
with open("../5データ/5-2評価実験用データ/5-2-1参考文献文字列_誤植なし（実在）/doi_list.txt","r") as f:
    lines = f.readlines()
eval_doi_list = [line.strip() for line in lines if line.strip()]

# # 日本学術会議登録団体のリスト
scj_registered_AcademicSocieties = []
with open("../5データ/5-3日本学術会議協力学術研究団体/scj_registered_AcademicSocieties.txt","r") as f:
    lines = f.readlines()
for line in lines:
    tmp = line.split(",")
    scj_registered_AcademicSocieties += tmp

print(f"DOI件数：{len(eval_doi_list)}件")

DOI件数：863件


In [3]:
# Cell 17
# 日本語を含むか判定
def has_japanese(text):
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

# 標題・著者・雑誌名に日本語があるか判定
def check_lang(doc):
    title_lang,creator_lang,journal_lang = "","",""
    # 表題の言語について
    for title in doc["title_list"]:
        lang = title.get("lang","")
        if lang == "":  # lang属性を持たない場合
            if has_japanese(title.get("title","")):
                lang = "ja"
        if lang == "ja":
            title_lang = "ja"
            break
    # 著者の言語について
    for creator in doc["creator_list"]:
        for creator_name in creator["names"]:
            lang = creator_name.get("lang","")
            if lang == "":  # lang属性を持たない場合
                if has_japanese(creator_name.get("first_name","")):
                    lang = "ja"
            if lang == "ja":
                creator_lang = "ja"
                break
    # 雑誌の言語について
    for journal in doc["journal_title_name_list"]:
        lang = journal.get("lang","")
        if lang == "":
            if has_japanese(journal.get("journal_title_name","")):
                lang = "ja"
        if lang == "ja":
            journal_lang = "ja"
            break
    return title_lang == "ja" and creator_lang == "ja" and journal_lang == "ja"                                        

# 学術会議登録団体であるか判定(計算量が多いの改善)
def check_publisher(doc,scj_registered_AcademicSocieties):
    flag = False
    for academicSocieties in scj_registered_AcademicSocieties:
        for publisher in doc["publisher_list"]:
            if academicSocieties in publisher.get("publisher_name",""):
                flag = True
    return flag            

# 標題の取得
def get_title(doc):
    title_text = ""
    for val in doc["title_list"]:
        lang,title,subtitle = val.get("lang",""),val.get("title",""),val.get("subtitle","")
        if lang == "":
            if has_japanese(title):
                lang = "ja"
        if lang == "ja":
            title_text = title
            if subtitle != "":
                title_text = title_text + " " + subtitle
    return title_text                   

# 雑誌名の取得
def get_journal(doc):
    journal_text = ""
    for val in doc["journal_title_name_list"]:
        journal_title,Type,lang = val.get("journal_title_name",""),val.get("type",""),val.get("lang","")
        if lang == "":
            if has_japanese(journal_title):
                lang = "ja"
        if lang == "ja":
            journal_text = journal_title
            if Type == "full":
                break
    return journal_text                

# 著者の取得
def get_creator(doc):
    creator_list = []
    for val in doc["creator_list"]:
        if len(val["names"]) == 1:  # 著者の表記が1通りの場合
            last_name,first_name = val["names"][0].get("last_name",""),val["names"][0].get("first_name","")
            creator_list.append([last_name,first_name])
        else:    # 著者の表記が2通り以上の場合
            for x in val["names"]:
                last_name,first_name,lang = x.get("last_name",""),x.get("first_name",""),x.get("lang","")
                if has_japanese(first_name):
                    lang = "ja"
                if lang == "ja":
                    creator_list.append([last_name,first_name])
                    break
    return creator_list            

# 参考文献文字列の作成(人工知能学会)
def create_reference_jsai(creator_list,title,journal,volume,issue,page_list,year):
    tmp = []
    for name in creator_list:
            if name[0] != "":
                if has_japanese(name[0]):
                    tmp.append(name[0])
                else:
                    tmp.append(name[0]+", "+name[1][0]+".")
            else:
                tmp.append(name[1])
    creator_text = "，".join(tmp)
    reference = creator_text + "：" + title + "，" + journal + "，" + "Vol." + volume + ", " + "No."\
    + issue + ", " + "pp." + page_list[0] + "-" + page_list[1] + " " + "(" + year + ")" + "."
    return reference

# 参考文献文字列の作成(情報処理学会)
def create_reference_ipsj(creator_list,title,journal,volume,issue,page_list,year,count):
    creator_text = ""
    tmp = []
    num = 0
    for name in creator_list:
        num += 1
        if num == 4:
            creator_text = "ほか"
            break
        else:
            if name[0] != "":
                if has_japanese(name[0]):
                    tmp.append(name[0] + name[1])
                else:
                    tmp.append(name[0]+", "+name[1][0]+".")
            else:
                tmp.append(name[1])
    if len(tmp) <= 3:
        creator_text = ""
    creator_text = "，".join(tmp) + creator_text
    reference = creator_text+"："+title+"，"+journal+"，"+"Vol."+volume+"，"+"No."+issue+"，"+"pp."+page_list[0]+"-"+\
    page_list[1]+"（"+year+"）"+"．"
    return reference         

# 参考文献文字列の作成(日本言語学会)
def create_reference_lsj(creator_list,title,journal,volume,issue,page_list,year):
    tmp = []
    for name in creator_list:
        tmp.append(name[0]+name[1])
    reference = "・".join(tmp)+"（"+year+"）"+"「"+title+"」"+"『"+journal+"』"+volume+"（"+issue+"）"+": "+page_list[0]+"–"+page_list[1]+"."
    return reference     

# 以下本処理

title_list = []
count = 1
reference_jsai_list = []
reference_ipsj_list = []
reference_lsj_list = []
document_num = 0

for doi in eval_doi_list:
    doc = mycol.find_one({"doi":doi},{"_id":0})
    if doc is None:
        raise ValueError(f"MongoDB に DOI が見つかりません: {doi}")
    if not check_lang(doc):
        raise ValueError(f"日本語条件を満たしません: {doi}")
    if not check_publisher(doc,scj_registered_AcademicSocieties):
        raise ValueError(f"学術会議登録団体条件を満たしません: {doi}")

    title = get_title(doc)
    journal = get_journal(doc)
    volume = doc["volume"]
    issue = doc["issue"]
    page_list = [doc["first_page"],doc["last_page"]]
    year = doc["publication_date"]["publication_year"]
    creator_list = get_creator(doc)
    document_num += 1

    reference_jsai = create_reference_jsai(creator_list.copy(),title,journal,volume,issue,page_list.copy(),year)
    reference_ipsj = create_reference_ipsj(creator_list.copy(),title,journal,volume,issue,page_list.copy(),year,count)
    reference_lsj = create_reference_lsj(creator_list.copy(),title,journal,volume,issue,page_list.copy(),year)
    count += 1

    reference_jsai_list.append(reference_jsai)
    reference_ipsj_list.append(reference_ipsj)
    reference_lsj_list.append(reference_lsj)
    title_list.append(title)
print(f"該当論文数：{len(reference_jsai_list)}件") 

# テキストファイルの作成
with open('../5データ/5-2評価実験用データ/5-2-1参考文献文字列_誤植なし（実在）/positive_reference_jsai_eval_none_typo.txt','w') as f:
    f.write('\n'.join(reference_jsai_list))
with open('../5データ/5-2評価実験用データ/5-2-1参考文献文字列_誤植なし（実在）/positive_reference_ipsj_eval_none_typo.txt','w') as f:
    f.write('\n'.join(reference_ipsj_list))
with open('../5データ/5-2評価実験用データ/5-2-1参考文献文字列_誤植なし（実在）/positive_reference_lsj_eval_none_typo.txt','w') as f:
    f.write('\n'.join(reference_lsj_list))
print("テキストファイル作成完了")                                              

該当論文数：863件
テキストファイル作成完了


### Cell 18
## ”参考文献文字列（架空）_評価実験用”の作成
* 予備実験とは異なるデータ

In [3]:
# Cell 19
# Geminiが生成した表題をリストにまとめる
with open('../5データ/5-2評価実験用データ/5-2-3論文タイトル一覧/llm_create_titles.txt','r') as f:
    lines = f.readlines()

# テキストファイルには改行コードが含まれているためそれを除去
cleaned_lines = []
for line in lines:
    cleaned_lines.append(line.strip())
llm_create_titles = []
for cleaned_line in cleaned_lines:
    llm_create_titles.append(cleaned_line)

for i in range(10):
    print(llm_create_titles[i])    

print(f"表題：{len(llm_create_titles)}件")

原子力災害後の旧避難区域における放射線教育と地域交流の意義と効果 福島県川俣町山木屋地区の交流活動を事例として
イノシシの新規生息域拡大における地域主体の捕獲体制構築法 海を越えた愛媛県中島本島へのイノシシの移入に着目
木の駅プロジェクトでの出荷者行動に関する分析
農業分野における太陽光発電の自家消費に関する考察
都市農業経営における生産緑地の貸借条件と活用実態
小規模水力発電導入に向けた民間企業と地域の連携形成プロセス
農林業センサスに基づく愛媛県臨海部の農業集落単位での地域特性の基礎研究
環境保全型農業が生産者と消費者の関係性に与える影響 福井県池田町の「ゆうきげんき正直農業」の事例
個体間類似性のグラフ表示法の提案と農業部門構成の地区間類似性分析への応用
中国貧困農村における観光業参画による農家の増収効果 河南省における観光扶貧重点村の農家調査に基づく実証
表題：863件


In [7]:
# Cell 20
# 参考文献文字列　評価実験　正例の作成クエリを実行している必要あり

# 日本語を含むか判定
def has_japanese(text):
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

# 標題・著者・雑誌名に日本語があるか判定
def check_lang(doc):
    title_lang,creator_lang,journal_lang = "","",""
    # 表題の言語について
    for title in doc["title_list"]:
        lang = title.get("lang","")
        if lang == "":  # lang属性を持たない場合
            if has_japanese(title.get("title","")):
                lang = "ja"
        if lang == "ja":
            title_lang = "ja"
            break
    # 著者の言語について
    for creator in doc["creator_list"]:
        for creator_name in creator["names"]:
            lang = creator_name.get("lang","")
            if lang == "":  # lang属性を持たない場合
                if has_japanese(creator_name.get("first_name","")):
                    lang = "ja"
            if lang == "ja":
                creator_lang = "ja"
                break
    # 雑誌の言語について
    for journal in doc["journal_title_name_list"]:
        lang = journal.get("lang","")
        if lang == "":
            if has_japanese(journal.get("journal_title_name","")):
                lang = "ja"
        if lang == "ja":
            journal_lang = "ja"
            break
    return title_lang == "ja" and creator_lang == "ja" and journal_lang == "ja"                                        

# 学術会議登録団体であるか判定(計算量が多いの改善)
def check_publisher(doc,scj_registered_AcademicSocieties):
    flag = False
    for academicSocieties in scj_registered_AcademicSocieties:
        for publisher in doc["publisher_list"]:
            if academicSocieties in publisher.get("publisher_name",""):
                flag = True
    return flag            

# 標題の取得
def get_title(doc):
    title_text = ""
    for val in doc["title_list"]:
        lang,title,subtitle = val.get("lang",""),val.get("title",""),val.get("subtitle","")
        if lang == "":
            if has_japanese(title):
                lang = "ja"
        if lang == "ja":
            title_text = title
            if subtitle != "":
                title_text = title_text + " " + subtitle
    return title_text                   

# 雑誌名の取得
def get_journal(doc):
    journal_text = ""
    for val in doc["journal_title_name_list"]:
        journal_title,Type,lang = val.get("journal_title_name",""),val.get("type",""),val.get("lang","")
        if lang == "":
            if has_japanese(journal_title):
                lang = "ja"
        if lang == "ja":
            journal_text = journal_title
            if Type == "full":
                break
    return journal_text                

# 著者の取得
def get_creator(doc):
    creator_list = []
    for val in doc["creator_list"]:
        if len(val["names"]) == 1:  # 著者の表記が1通りの場合
            last_name,first_name = val["names"][0].get("last_name",""),val["names"][0].get("first_name","")
            creator_list.append([last_name,first_name])
        else:    # 著者の表記が2通り以上の場合
            for x in val["names"]:
                last_name,first_name,lang = x.get("last_name",""),x.get("first_name",""),x.get("lang","")
                if has_japanese(first_name):
                    lang = "ja"
                if lang == "ja":
                    creator_list.append([last_name,first_name])
                    break
    return creator_list            

# 学会名の取得
def get_publisher(doc,scj_registered_AcademicSocieties):
    publisher_name = ""
    for academicSocieties in scj_registered_AcademicSocieties:
        finish_flag = False
        for publisher in doc["publisher_list"]:
            if academicSocieties in publisher.get("publisher_name",""):
                publisher_name = academicSocieties
                finish_flag = True
        if finish_flag:
            break
    return publisher_name            

# 参考文献文字列の作成(人工知能学会)
def create_reference_jsai(creator_list,llm_create_title,publisher_name,year):
    tmp = []
    for name in creator_list:
        if name[0] != "":
            if has_japanese(name[0]):
                tmp.append(name[0])
            else:
                tmp.append(name[0]+", "+name[1][0]+".")
        else:
            tmp.append(name[1])                
    if len(tmp) != 1:   # 著者を一名捨てる
        tmp.pop(-1)            
    creator_text = "，".join(tmp)
    year = int(year) - 1
    reference = creator_text + "：" + llm_create_title + "，" + publisher_name + " " + "(" + str(year) + ")" + "."
    return reference

# 参考文献文字列の作成(情報処理学会)
def create_reference_ipsj(creator_list,llm_create_title,publisher_name,year):
    creator_text = ""
    tmp = []
    num = 0
    for name in creator_list:
        num += 1
        if num == 5:
            creator_text = "ほか"
            break
        else:
            if name[0] != "": 
                if has_japanese(name[0]):
                    tmp.append(name[0] + name[1])
                else:
                    tmp.append(name[0]+", "+name[1][0]+".")
            else:
                tmp.append(name[1])
    if len(tmp) != 1:   # 著者一名を削除
        tmp.pop(-1)                    
    creator_text = "，".join(tmp) + creator_text
    year = int(year) - 1
    reference = creator_text+"："+llm_create_title+"，"+publisher_name+" "+"（"+str(year)+"）"+"．"
    return reference         

# 参考文献文字列の作成(日本言語学会)
def create_reference_lsj(creator_list,llm_create_title,publisher,year):
    tmp = []
    for name in creator_list:
        tmp.append(name[0]+name[1])
    if len(tmp) != 1:
        tmp.pop(-1)
    year = int(year) - 1        
    reference = "・".join(tmp)+"（"+str(year)+"）"+"「"+llm_create_title+"」"+"『"+publisher+"』"
    return reference     

# 以下本処理

documents = mycol.find(query)
count = 0
negative_examples_num = 0
reference_jsai_list = []
reference_ipsj_list = []
reference_lsj_list = []

for doc in documents:
    if check_lang(doc) and check_publisher(doc,scj_registered_AcademicSocieties):
        llm_create_title = llm_create_titles[count]
        title = get_title(doc)
        journal = get_journal(doc)
        volume = doc["volume"]
        issue = doc["issue"]
        page_list = [doc["first_page"],doc["last_page"]]
        year = doc["publication_date"]["publication_year"]
        creator_list = get_creator(doc)
        publisher_name = get_publisher(doc,scj_registered_AcademicSocieties)
        
        #reference_jsai = create_reference_jsai(creator_list,llm_create_title,publisher_name,year)
        #reference_ipsj = create_reference_ipsj(creator_list,llm_create_title,publisher_name,year)
        #reference_lsj = create_reference_lsj(creator_list,title,journal,volume,issue,page_list,year)

        if title != llm_create_title:   # Geminiによる変換が上手くいっているもの [if llm_create_title not in title:]に変換したほうがいいかも
            negative_examples_num += 1
            reference_jsai = create_reference_jsai(creator_list,llm_create_title,publisher_name,year)
            reference_ipsj = create_reference_ipsj(creator_list,llm_create_title,publisher_name,year)
            reference_lsj = create_reference_lsj(creator_list,llm_create_title,publisher_name,year)
            reference_jsai_list.append(reference_jsai)
            reference_ipsj_list.append(reference_ipsj)
            reference_lsj_list.append(reference_lsj)
        count += 1

print(f"該当件数：{len(reference_ipsj_list)}件")
# テキストファイルの作成        
with open('../5データ/5-2評価実験用データ/5-2-2参考文献文字列（架空）/negative_reference_jsai_eval.txt','x') as f:
    f.write('\n'.join(reference_jsai_list))
with open('../5データ/5-2評価実験用データ/5-2-2参考文献文字列（架空）/negative_reference_ipsj_eval.txt','x') as f:
    f.write('\n'.join(reference_ipsj_list))
with open('../5データ/5-2評価実験用データ/5-2-2参考文献文字列（架空）/negative_reference_lsj_eval.txt','x') as f:
    f.write('\n'.join(reference_lsj_list))
print("テキストファイル作成完了")            
                                              

該当件数：818件
テキストファイル作成完了


### Cell 21
### その他

In [3]:
# Cell 22
# Geminiが生成した表題をリストにまとめる
with open('../5データ/5-2評価実験用データ/5-2-3論文タイトル一覧/llm_create_titles.txt','r') as f:
    lines = f.readlines()

# テキストファイルには改行コードが含まれているためそれを除去
cleaned_lines = []
for line in lines:
    cleaned_lines.append(line.strip())
llm_create_titles = []
for cleaned_line in cleaned_lines:
    llm_create_titles.append(cleaned_line)

with open('../5データ/5-2評価実験用データ/5-2-3論文タイトル一覧/eval_experiment_titles.txt','r') as f:
    lines = f.readlines()
cleaned_lines = []
for line in lines:
    cleaned_lines.append(line.strip())
titles = []
for cleaned_line in cleaned_lines:
    titles.append(cleaned_line)

count = 0
for i in range(len(titles)):
    if titles[i] != llm_create_titles[i]:
        count += 1
print(count)        

        

818
